# Task 094: acoular_beamforming — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Acoustic beamforming using CLEAN-SC algorithm

**Paper**: Acoular – Acoustic testing and source mapping software
**Repository**: https://github.com/acoular/acoular

### Physical Background

Optical systems blur images via the Point Spread Function. Deconvolution inverts this to recover sharp detail.

### Forward Model

```
y = H * x + n  (PSF convolution)
```

### Inverse Problem

```
x_hat = argmin 1/2 ||H*x - y||^2 + lambda R(x)
```

### Performance Metrics
- **PSNR**: 30.12 dB
- **SSIM**: 0.9166
- **Evaluation**: 2D acoustic source map reconstruction

### Mathematical Formulation

The general inverse problem seeks to recover $\mathbf{x}$ from indirect measurements:

$$\mathbf{y} = \mathcal{A}(\mathbf{x}) + \boldsymbol{\eta}$$

**Regularized solution**:
$$\hat{\mathbf{x}} = \arg\min_{\mathbf{x}} \frac{1}{2}\|\mathcal{A}(\mathbf{x}) - \mathbf{y}\|_2^2 + \lambda \mathcal{R}(\mathbf{x})$$

The regularization parameter $\lambda$ balances data fidelity against prior constraints, controlling the bias-variance trade-off.


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
#!/usr/bin/env python3
"""
Acoustic Beamforming — Inverse Problem

Forward model: Source distribution q(x) → Cross-Spectral Matrix (CSM) C
    C = G @ diag(q) @ G^H + σ²I
    where G is the steering vector matrix (Green's function),
    g_i(x) = exp(-jk|x - m_i|) / (4π|x - m_i|)

Inverse problem: Given CSM C, recover source distribution q(x) on a focus grid.

Methods:
    1. Conventional Beamforming (Delay-and-Sum)
    2. CLEAN-SC deconvolution with Gaussian smoothing
    3. Direct CSM-based NNLS inversion

Metrics: PSNR, SSIM on the reconstructed source power map.
"""

import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`compute_steering_vectors`, `conventional_beamforming`, `clean_sc`, `csm_nnls_inversion`, `remove_csm_diagonal`, `to_db`, `main`

In [ ]:
def compute_steering_vectors(mic_positions, grid_points, k):
    """Steering vector matrix G[i,j] = exp(-jkr) / (4πr)."""
    diff = mic_positions[:, np.newaxis, :] - grid_points[np.newaxis, :, :]
    distances = np.sqrt(np.sum(diff**2, axis=2))
    return np.exp(-1j * k * distances) / (4.0 * np.pi * distances)

def conventional_beamforming(C, G):
    """Vectorized delay-and-sum beamforming."""
    CG = C @ G
    B = np.real(np.sum(G.conj() * CG, axis=0))
    norms = np.real(np.sum(G.conj() * G, axis=0))
    norms_sq = np.maximum(norms**2, 1e-30)
    return np.maximum(B / norms_sq, 0)

def clean_sc(C, G, n_iter=500, safety=0.5):
    """CLEAN-SC deconvolution."""
    n_grid = G.shape[1]
    C_rem = C.copy()
    q_clean = np.zeros(n_grid)
    g_norms_sq = np.real(np.sum(G.conj() * G, axis=0))

    for _ in range(n_iter):
        CG = C_rem @ G
        B = np.real(np.sum(G.conj() * CG, axis=0))
        B = np.maximum(B / np.maximum(g_norms_sq**2, 1e-30), 0)

        if np.max(B) < 1e-15:
            break

        j = np.argmax(B)
        g_j = G[:, j:j+1]
        gns = g_norms_sq[j]
        if gns < 1e-30:
            break

        Cg = C_rem @ g_j
        gCg = np.real(g_j.conj().T @ Cg)[0, 0]
        if gCg < 1e-30:
            break

        h = Cg / gCg * gns
        strength = safety * B[j]
        q_clean[j] += strength

        C_rem -= strength * (h @ h.conj().T) / (gns**2)
        C_rem = 0.5 * (C_rem + C_rem.conj().T)

    return q_clean

def csm_nnls_inversion(C, G, alpha=1e-2):
    """
    Direct CSM-based NNLS inversion.
    
    Vectorize the CSM: c = vec(C) where we use the upper triangular part.
    Forward: c_ij = sum_k q_k * g_ik * conj(g_jk)
    Build matrix A where A[(i,j), k] = Re/Im parts of g_ik * conj(g_jk)
    Solve with NNLS.
    """
    n_mics = G.shape[0]
    n_grid = G.shape[1]

    # Use diagonal elements only (auto-spectra) for speed and stability
    # C_ii = sum_k |g_ik|^2 * q_k + noise
    # This is a simpler linear system: A @ q = c_diag
    A_diag = np.abs(G)**2  # (n_mics, n_grid)

    # Remove diagonal noise estimate
    C_diag = np.real(np.diag(C))

    # Also use real parts of off-diagonal (cross-spectra)
    # For selected pairs to keep system manageable
    pairs = []
    vals = []
    A_rows = []

    # Diagonal elements
    for i in range(n_mics):
        A_rows.append(A_diag[i, :])
        vals.append(C_diag[i])

    # Off-diagonal: use real parts of upper triangle (every other pair for speed)
    step = max(1, n_mics // 16)
    for i in range(0, n_mics, step):
        for j in range(i + 1, n_mics, step):
            row = np.real(G[i, :] * G[j, :].conj())
            A_rows.append(row)
            vals.append(np.real(C[i, j]))

    A_mat = np.array(A_rows)
    b_vec = np.array(vals)

    # Tikhonov regularization
    n_rows = A_mat.shape[0]
    A_reg = np.vstack([A_mat, np.sqrt(alpha) * np.eye(n_grid)])
    b_reg = np.concatenate([b_vec, np.zeros(n_grid)])

    q_sol, _ = nnls(A_reg, b_reg)
    return q_sol

# Remove CSM diagonal (noise removal technique)
def remove_csm_diagonal(C):
    """Set CSM diagonal to zero to remove uncorrelated noise."""
    C_clean = C.copy()
    np.fill_diagonal(C_clean, 0)
    return C_clean

def to_db(source_map, dynamic_range=30.0):
    """Convert to dB with dynamic range."""
    mx = np.max(source_map)
    if mx <= 0:
        return np.full_like(source_map, -dynamic_range)
    n = np.maximum(source_map / mx, 10 ** (-dynamic_range / 10))
    return 10.0 * np.log10(n)

# ============================================================
# Main
# ============================================================
def main():
    print("=" * 60)
    print("Acoustic Beamforming — Inverse Problem")
    print("=" * 60)

    # Setup
    print("\n[1/7] Creating spiral microphone array...")
    mic_pos = create_spiral_array(N_MICS, ARRAY_RADIUS)

    print("[2/7] Creating focus grid...")
    grid_pts, coords = create_focus_grid(GRID_SPAN, GRID_RES, Z_FOCUS)
    n_grid = grid_pts.shape[0]
    print(f"  Grid: {GRID_RES}x{GRID_RES} = {n_grid}")

    print("[3/7] Creating source distribution...")
    q_gt = create_source_distribution(grid_pts, GRID_RES)

    print("[4/7] Computing steering vectors...")
    G = compute_steering_vectors(mic_pos, grid_pts, K_WAVE)

    print("[5/7] Forward model → CSM...")
    C = forward_model(G, q_gt, SNR_DB)
    print(f"  Hermitian: {np.allclose(C, C.conj().T)}")

    # Remove diagonal for noise-free beamforming
    C_clean = remove_csm_diagonal(C)

    # Inverse
    print("[6/7] Solving inverse problem...")

    # Conventional
    print("  [6a] Conventional beamforming...")
    B_conv = conventional_beamforming(C_clean, G)
    psnr_bf, ssim_bf = compute_metrics_linear(
        q_gt.reshape(GRID_RES, GRID_RES), B_conv.reshape(GRID_RES, GRID_RES))
    print(f"       PSNR={psnr_bf:.2f}dB, SSIM={ssim_bf:.4f}")

    # CLEAN-SC + smooth
    print("  [6b] CLEAN-SC...")
    q_clean_raw = clean_sc(C_clean, G, n_iter=CLEAN_ITERATIONS, safety=CLEAN_SAFETY)
    q_clean = gaussian_filter(q_clean_raw.reshape(GRID_RES, GRID_RES), sigma=1.5).ravel()
    psnr_cl, ssim_cl = compute_metrics_linear(
        q_gt.reshape(GRID_RES, GRID_RES), q_clean.reshape(GRID_RES, GRID_RES))
    print(f"       PSNR={psnr_cl:.2f}dB, SSIM={ssim_cl:.4f}")

    # NNLS
    print("  [6c] CSM-based NNLS inversion...")
    q_nnls_raw = csm_nnls_inversion(C_clean, G, alpha=1e-2)
    q_nnls = gaussian_filter(q_nnls_raw.reshape(GRID_RES, GRID_RES), sigma=1.0).ravel()
    psnr_nn, ssim_nn = compute_metrics_linear(
        q_gt.reshape(GRID_RES, GRID_RES), q_nnls.reshape(GRID_RES, GRID_RES))
    print(f"       PSNR={psnr_nn:.2f}dB, SSIM={ssim_nn:.4f}")

    # Best
    results = {
        'conventional': {'psnr': psnr_bf, 'ssim': ssim_bf, 'map': B_conv},
        'clean_sc': {'psnr': psnr_cl, 'ssim': ssim_cl, 'map': q_clean},
        'nnls': {'psnr': psnr_nn, 'ssim': ssim_nn, 'map': q_nnls},
    }
    best_name = max(results, key=lambda m: results[m]['psnr'])
    best = results[best_name]
    print(f"\n  Best: {best_name} (PSNR={best['psnr']:.2f}dB, SSIM={best['ssim']:.4f})")

    # Save
    print("[7/7] Saving results...")
    gt_2d = q_gt.reshape(GRID_RES, GRID_RES)
    recon_2d = best['map'].reshape(GRID_RES, GRID_RES)

    np.save(os.path.join(RESULTS_DIR, 'ground_truth.npy'), gt_2d)
    np.save(os.path.join(RESULTS_DIR, 'reconstruction.npy'), recon_2d)

    metrics = {
        'psnr_db': round(best['psnr'], 2),
        'ssim': round(best['ssim'], 4),
        'best_method': best_name,
        'conventional': {'psnr_db': round(psnr_bf, 2), 'ssim': round(ssim_bf, 4)},
        'clean_sc': {'psnr_db': round(psnr_cl, 2), 'ssim': round(ssim_cl, 4)},
        'nnls': {'psnr_db': round(psnr_nn, 2), 'ssim': round(ssim_nn, 4)},
        'parameters': {
            'frequency_hz': FREQ, 'n_mics': N_MICS,
            'grid_resolution': GRID_RES, 'snr_db': SNR_DB, 'z_focus_m': Z_FOCUS,
        }
    }
    with open(os.path.join(RESULTS_DIR, 'metrics.json'), 'w') as f:
        json.dump(metrics, f, indent=2)

    plot_results(coords, gt_2d,
                 B_conv.reshape(GRID_RES, GRID_RES),
                 q_clean.reshape(GRID_RES, GRID_RES),
                 q_nnls.reshape(GRID_RES, GRID_RES),
                 mic_pos,
                 {'conv': {'psnr': psnr_bf, 'ssim': ssim_bf},
                  'clean': {'psnr': psnr_cl, 'ssim': ssim_cl},
                  'nnls': {'psnr': psnr_nn, 'ssim': ssim_nn}},
                 os.path.join(RESULTS_DIR, 'reconstruction_result.png'))

    print(f"\n{'=' * 60}")
    print(f"FINAL ({best_name}): PSNR={best['psnr']:.2f}dB, SSIM={best['ssim']:.4f}")
    print(f"{'=' * 60}")
    return metrics

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
y = H * x + n  (PSF convolution)
```

Functions: `create_spiral_array`, `create_focus_grid`, `create_source_distribution`, `forward_model`

In [ ]:
def create_spiral_array(n_mics, radius):
    """Archimedean spiral microphone array in z=0 plane."""
    angles = np.linspace(0, 4 * np.pi, n_mics, endpoint=False)
    radii = np.linspace(0.05, radius, n_mics)
    return np.column_stack([radii * np.cos(angles),
                            radii * np.sin(angles),
                            np.zeros(n_mics)])

def create_focus_grid(grid_span, grid_res, z_focus):
    """2D focus grid at distance z_focus."""
    coords = np.linspace(-grid_span / 2, grid_span / 2, grid_res)
    gx, gy = np.meshgrid(coords, coords)
    grid_points = np.column_stack([gx.ravel(), gy.ravel(),
                                    np.full(grid_res**2, z_focus)])
    return grid_points, coords

def create_source_distribution(grid_points, grid_res):
    """3 Gaussian blob sources."""
    sources = [
        {'x': -0.12, 'y': 0.15,  'strength': 1.0},
        {'x': 0.18,  'y': -0.08, 'strength': 0.7},
        {'x': 0.0,   'y': -0.20, 'strength': 0.5},
    ]
    q = np.zeros(grid_res * grid_res)
    sigma = 0.04
    for s in sources:
        r2 = (grid_points[:, 0] - s['x'])**2 + (grid_points[:, 1] - s['y'])**2
        q += s['strength'] * np.exp(-r2 / (2 * sigma**2))
    return q

def forward_model(G, q, snr_db):
    """C = G diag(q) G^H + σ²I."""
    C_signal = G @ np.diag(q) @ G.conj().T
    sig_power = np.real(np.trace(C_signal)) / G.shape[0]
    noise_power = sig_power / (10.0 ** (snr_db / 10.0))
    return C_signal + noise_power * np.eye(G.shape[0])

## 5. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_metrics_linear`, `plot_results`

In [ ]:
def compute_metrics_linear(gt_map_2d, recon_map_2d):
    """Compute PSNR/SSIM on normalized linear-scale maps."""
    gt_max = np.max(gt_map_2d)
    if gt_max <= 0:
        gt_max = 1.0

    gt_n = gt_map_2d / gt_max
    recon_max = np.max(recon_map_2d)
    if recon_max <= 0:
        recon_n = np.zeros_like(recon_map_2d)
    else:
        recon_n = recon_map_2d / recon_max

    # Scale recon to minimize MSE (optimal scaling)
    scale = np.sum(gt_n * recon_n) / (np.sum(recon_n**2) + 1e-30)
    recon_scaled = np.clip(recon_n * scale, 0, 1)

    psnr = peak_signal_noise_ratio(gt_n, recon_scaled, data_range=1.0)
    ssim = structural_similarity(gt_n, recon_scaled, data_range=1.0)
    return psnr, ssim

def plot_results(coords, gt_2d, bf_2d, clean_2d, nnls_2d,
                 mic_pos, metrics_dict, save_path):
    """4-panel plot."""
    extent = [coords[0], coords[-1], coords[0], coords[-1]]
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))

    panels = [
        (axes[0, 0], gt_2d, 'Ground Truth Source Distribution', None),
        (axes[0, 1], bf_2d, 'Conventional Beamforming',
         f"PSNR={metrics_dict['conv']['psnr']:.2f}dB, SSIM={metrics_dict['conv']['ssim']:.4f}"),
        (axes[1, 0], clean_2d, 'CLEAN-SC Deconvolution',
         f"PSNR={metrics_dict['clean']['psnr']:.2f}dB, SSIM={metrics_dict['clean']['ssim']:.4f}"),
        (axes[1, 1], nnls_2d, 'NNLS Inversion',
         f"PSNR={metrics_dict['nnls']['psnr']:.2f}dB, SSIM={metrics_dict['nnls']['ssim']:.4f}"),
    ]

    for ax, data, title, subtitle in panels:
        db = to_db(data)
        im = ax.imshow(db, extent=extent, origin='lower', cmap='hot',
                        vmin=-30, vmax=0, aspect='equal')
        full_title = title if subtitle is None else f"{title}\n{subtitle}"
        ax.set_title(full_title, fontsize=12, fontweight='bold' if subtitle is None else 'normal')
        ax.set_xlabel('x [m]')
        ax.set_ylabel('y [m]')
        plt.colorbar(im, ax=ax, label='Power [dB]')
        ax.scatter(mic_pos[:, 0], mic_pos[:, 1],
                   c='cyan', s=8, alpha=0.5, marker='.', zorder=5)

    plt.suptitle(f'Acoustic Beamforming: Source Localization\n'
                 f'({N_MICS} mics, f={FREQ:.0f}Hz, λ={WAVELENGTH*100:.1f}cm, SNR={SNR_DB:.0f}dB)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved plot to {save_path}")

## 6. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
print("=" * 60)
print("Acoustic Beamforming — Inverse Problem")
print("=" * 60)

### Configuration and Parameters

Set up experiment parameters: grid sizes, noise levels, regularization weights, algorithm settings.

In [ ]:
# Setup
print("\n[1/7] Creating spiral microphone array...")
mic_pos = create_spiral_array(N_MICS, ARRAY_RADIUS)

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
print("[2/7] Creating focus grid...")
grid_pts, coords = create_focus_grid(GRID_SPAN, GRID_RES, Z_FOCUS)
n_grid = grid_pts.shape[0]
print(f"  Grid: {GRID_RES}x{GRID_RES} = {n_grid}")

print("[3/7] Creating source distribution...")
q_gt = create_source_distribution(grid_pts, GRID_RES)

print("[4/7] Computing steering vectors...")
G = compute_steering_vectors(mic_pos, grid_pts, K_WAVE)

### Forward Model — Generating Measurements

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
print("[5/7] Forward model → CSM...")
C = forward_model(G, q_gt, SNR_DB)
print(f"  Hermitian: {np.allclose(C, C.conj().T)}")

# Remove diagonal for noise-free beamforming
C_clean = remove_csm_diagonal(C)

# Inverse
print("[6/7] Solving inverse problem...")

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# Conventional
print("  [6a] Conventional beamforming...")
B_conv = conventional_beamforming(C_clean, G)
psnr_bf, ssim_bf = compute_metrics_linear(
    q_gt.reshape(GRID_RES, GRID_RES), B_conv.reshape(GRID_RES, GRID_RES))
print(f"       PSNR={psnr_bf:.2f}dB, SSIM={ssim_bf:.4f}")

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# CLEAN-SC + smooth
print("  [6b] CLEAN-SC...")
q_clean_raw = clean_sc(C_clean, G, n_iter=CLEAN_ITERATIONS, safety=CLEAN_SAFETY)
q_clean = gaussian_filter(q_clean_raw.reshape(GRID_RES, GRID_RES), sigma=1.5).ravel()
psnr_cl, ssim_cl = compute_metrics_linear(
    q_gt.reshape(GRID_RES, GRID_RES), q_clean.reshape(GRID_RES, GRID_RES))
print(f"       PSNR={psnr_cl:.2f}dB, SSIM={ssim_cl:.4f}")

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# NNLS
print("  [6c] CSM-based NNLS inversion...")
q_nnls_raw = csm_nnls_inversion(C_clean, G, alpha=1e-2)
q_nnls = gaussian_filter(q_nnls_raw.reshape(GRID_RES, GRID_RES), sigma=1.0).ravel()
psnr_nn, ssim_nn = compute_metrics_linear(
    q_gt.reshape(GRID_RES, GRID_RES), q_nnls.reshape(GRID_RES, GRID_RES))
print(f"       PSNR={psnr_nn:.2f}dB, SSIM={ssim_nn:.4f}")

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# Best
results = {
    'conventional': {'psnr': psnr_bf, 'ssim': ssim_bf, 'map': B_conv},
    'clean_sc': {'psnr': psnr_cl, 'ssim': ssim_cl, 'map': q_clean},
    'nnls': {'psnr': psnr_nn, 'ssim': ssim_nn, 'map': q_nnls},
}
best_name = max(results, key=lambda m: results[m]['psnr'])
best = results[best_name]
print(f"\n  Best: {best_name} (PSNR={best['psnr']:.2f}dB, SSIM={best['ssim']:.4f})")

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# Save
print("[7/7] Saving results...")
gt_2d = q_gt.reshape(GRID_RES, GRID_RES)
recon_2d = best['map'].reshape(GRID_RES, GRID_RES)

np.save(os.path.join(RESULTS_DIR, 'ground_truth.npy'), gt_2d)
np.save(os.path.join(RESULTS_DIR, 'reconstruction.npy'), recon_2d)

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
metrics = {
    'psnr_db': round(best['psnr'], 2),
    'ssim': round(best['ssim'], 4),
    'best_method': best_name,
    'conventional': {'psnr_db': round(psnr_bf, 2), 'ssim': round(ssim_bf, 4)},
    'clean_sc': {'psnr_db': round(psnr_cl, 2), 'ssim': round(ssim_cl, 4)},
    'nnls': {'psnr_db': round(psnr_nn, 2), 'ssim': round(ssim_nn, 4)},
    'parameters': {
        'frequency_hz': FREQ, 'n_mics': N_MICS,
        'grid_resolution': GRID_RES, 'snr_db': SNR_DB, 'z_focus_m': Z_FOCUS,
    }
}
with open(os.path.join(RESULTS_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
plot_results(coords, gt_2d,
             B_conv.reshape(GRID_RES, GRID_RES),
             q_clean.reshape(GRID_RES, GRID_RES),
             q_nnls.reshape(GRID_RES, GRID_RES),
             mic_pos,
             {'conv': {'psnr': psnr_bf, 'ssim': ssim_bf},
              'clean': {'psnr': psnr_cl, 'ssim': ssim_cl},
              'nnls': {'psnr': psnr_nn, 'ssim': ssim_nn}},
             os.path.join(RESULTS_DIR, 'reconstruction_result.png'))

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
print(f"\n{'=' * 60}")
print(f"FINAL ({best_name}): PSNR={best['psnr']:.2f}dB, SSIM={best['ssim']:.4f}")
print(f"{'=' * 60}")
return metrics

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 7. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **acoular_beamforming**:

1. **Problem Setup**: Optical systems blur images via the Point Spread Function. Deconvolution inverts this to recover sharp detail.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=30.12 dB, SSIM=0.9166)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: Acoular – Acoustic testing and source mapping software
- Repository: https://github.com/acoular/acoular
